# WeightedKgBlend — Update MIND Indication Edges

Adds new FDA-approved drug-disease indication edges to MIND by matching
DrugCentral indications (live PostgreSQL) to MIND nodes using identifier cascades.

**Matching strategy (in order of reliability):**
1. CHEBI ID (drug) + DOID (disease)
2. UNII ID (drug) + DOID (disease)
3. CHEBI/UNII (drug) + MESH ID (disease, via doid_xref)

**Result:** +1,391 new edges (5,254 → 6,645 total indication edges)

In [ ]:
import pandas as pd
import psycopg2
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

DATA = Path('/Users/meghamala/projects/WeightedKgBlend/data')

# ── Load MIND ─────────────────────────────────────────────────────────────
mind = pd.read_csv(DATA / 'mind.tsv', sep='\t', header=None, names=['h', 'r', 't'])
print(f'MIND triples: {len(mind):,}')
print(f'Existing indication edges: {(mind["r"]=="indication").sum():,}')

# Index existing indications
existing = set(zip(
    mind[mind['r'] == 'indication']['h'],
    mind[mind['r'] == 'indication']['t']
))

# Index MIND nodes by prefix
mind_nodes = set(mind['h'].tolist() + mind['t'].tolist())
mind_chebi = {n for n in mind_nodes if n.startswith('CHEBI:')}
mind_unii  = {n.replace('UNII:', '') for n in mind_nodes if n.startswith('UNII:')}
mind_doids = {n for n in mind_nodes if n.startswith('DOID:')}
mind_mesh  = {n for n in mind_nodes if n.startswith('MESH:')}

print(f'\nMIND drug nodes  — CHEBI: {len(mind_chebi):,}  UNII: {len(mind_unii):,}')
print(f'MIND disease nodes — DOID: {len(mind_doids):,}  MESH: {len(mind_mesh):,}')

In [ ]:
# ── Connect to DrugCentral public PostgreSQL ───────────────────────────────
conn = psycopg2.connect(
    host='unmtid-dbs.net', port=5433, dbname='drugcentral',
    user='drugman', password='dosage', connect_timeout=15
)
print('Connected to DrugCentral')

# ── Query indications with CHEBI, UNII drug IDs and DOID disease IDs ──────
query = """
SELECT
    s.name           AS drug_name,
    chebi.identifier AS chebi_id,
    unii.identifier  AS unii_id,
    o.concept_name   AS disease_name,
    o.umls_cui,
    o.doid,
    dx.xref          AS disease_mesh
FROM omop_relationship_doid_view o
JOIN structures s ON s.id = o.struct_id
LEFT JOIN identifier chebi ON chebi.struct_id = o.struct_id AND chebi.id_type = 'CHEBI'
LEFT JOIN identifier unii  ON unii.struct_id  = o.struct_id AND unii.id_type  = 'UNII'
LEFT JOIN doid_xref dx     ON dx.doid = o.doid AND dx.source = 'MESH'
WHERE o.relationship_name = 'indication'
"""
dc = pd.read_sql(query, conn)
conn.close()

print(f'DrugCentral indications fetched: {len(dc):,}')
print(f'  with CHEBI : {dc["chebi_id"].notna().sum():,}')
print(f'  with UNII  : {dc["unii_id"].notna().sum():,}')
print(f'  with DOID  : {dc["doid"].notna().sum():,}')
print(f'  with disease MESH: {dc["disease_mesh"].notna().sum():,}')

In [ ]:
# ── Match DrugCentral → MIND nodes ────────────────────────────────────────
new_edges = []
seen = set()
match_counts = {'CHEBI+DOID': 0, 'UNII+DOID': 0, 'CHEBI+MESH': 0, 'UNII+MESH': 0}

for _, row in dc.iterrows():
    # ── Drug node ─────────────────────────────────────────────────────────
    drug_node = None
    if pd.notna(row['chebi_id']):
        n = row['chebi_id'] if ':' in str(row['chebi_id']) else f"CHEBI:{row['chebi_id']}"
        if n in mind_chebi:
            drug_node = n
    if not drug_node and pd.notna(row['unii_id']):
        n = f"UNII:{row['unii_id']}"
        if n in mind_nodes:
            drug_node = n

    # ── Disease node ──────────────────────────────────────────────────────
    dis_node = None
    dis_method = None
    if pd.notna(row['doid']) and row['doid'] in mind_doids:
        dis_node   = row['doid']
        dis_method = 'DOID'
    elif pd.notna(row['disease_mesh']):
        n = f"MESH:{row['disease_mesh']}"
        if n in mind_mesh:
            dis_node   = n
            dis_method = 'MESH'

    if drug_node and dis_node:
        pair = (drug_node, dis_node)
        if pair not in existing and pair not in seen:
            seen.add(pair)
            drug_method = 'CHEBI' if 'CHEBI' in drug_node else 'UNII'
            match_counts[f'{drug_method}+{dis_method}'] += 1
            new_edges.append({
                'head'        : drug_node,
                'relation'    : 'indication',
                'tail'        : dis_node,
                'drug_name'   : row['drug_name'],
                'disease_name': row['disease_name'],
                'match_method': f'{drug_method}+{dis_method}',
            })

new_df = pd.DataFrame(new_edges)

print(f'New indication edges found: {len(new_df):,}')
print(f'Total after update        : {len(existing) + len(new_df):,}')
print(f'\nMatch method breakdown:')
for method, count in match_counts.items():
    print(f'  {method}: {count}')

In [ ]:
# ── Sample new edges ──────────────────────────────────────────────────────
print('Sample new indication edges:')
print(new_df[['drug_name', 'disease_name', 'head', 'tail', 'match_method']]
      .head(15).to_string(index=False))

In [ ]:
# ── Save mapping report and updated MIND ──────────────────────────────────
new_df.to_csv(DATA / 'mapping_report.csv', index=False)
print(f'Saved mapping report: {len(new_df):,} new edges → data/mapping_report.csv')

# Append new triples to MIND
new_triples = new_df[['head', 'relation', 'tail']].copy()
new_triples.columns = [0, 1, 2]
mind_updated = pd.concat([mind, new_triples], ignore_index=True)
mind_updated.to_csv(DATA / 'mind_updated.tsv', sep='\t', index=False, header=False)

print(f'Saved mind_updated.tsv: {len(mind_updated):,} total edges')
print()
print('=' * 50)
print('MANUSCRIPT NUMBERS')
print('=' * 50)
print(f'Original MIND indication edges : {len(existing):,}')
print(f'New edges from DrugCentral     : +{len(new_df):,}')
print(f'Updated total                  : {len(existing) + len(new_df):,}')
print(f'Improvement                    : +{len(new_df)/len(existing)*100:.1f}%')